In [ ]:
# =====================================================================
# MT2KD - chunk long recordings into short clips for the web app (ONE CELL)
# Runtime: CPU is enough (no GPU needed).
# Input : audio files in your Drive folder (INPUT_DIR)
# Output: 16 kHz mono WAV clips + one zip, saved in Drive (OUTPUT_DIR)
#         Only NEW files are created; nothing is deleted or overwritten outside OUTPUT_DIR.
# =====================================================================

# ----------------------------- SETTINGS ------------------------------
INPUT_DIR   = "/content/drive/MyDrive/MT2KD/sanada_data"     # <- your uploaded dataset folder
OUTPUT_DIR  = "/content/drive/MyDrive/MT2KD/sanada_chunks"   # <- clips are saved here
CLIP_SECONDS_LONG  = 30   # long voice recordings (> LONG_FILE_SECONDS) -> ~30 s clips (ASR)
CLIP_SECONDS_SHORT = 10   # also make ~10 s clips of long recordings (AT / SV tests)
LONG_FILE_SECONDS  = 60   # files longer than this are chunked; shorter ones are just converted
MIN_CLIP_SECONDS   = 3    # drop clips shorter than this
SPEAKER_NAME       = "Sanada"   # 5 x 10 s clips for SV "Registered speaker" (set None to skip)
MAKE_ZIP           = True
# ---------------------------------------------------------------------

import os, sys, time, zipfile
from pathlib import Path
t0 = time.time()

# STEP 1: mount Drive
print("▶ STEP 1/5: mounting Google Drive ...")
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
print("✅ STEP 1 COMPLETED: Drive mounted")

# STEP 2: libraries (preinstalled on Colab)
print("▶ STEP 2/5: checking libraries ...")
import numpy as np
import soundfile as sf
import librosa
SR = 16000
AUDIO_EXT = {".wav", ".flac", ".mp3", ".ogg", ".m4a", ".aac", ".wma", ".opus", ".webm", ".mp4"}
print(f"✅ STEP 2 COMPLETED: numpy {np.__version__}, soundfile {sf.__version__}, librosa {librosa.__version__}")

# STEP 3: find audio files
print("▶ STEP 3/5: looking for audio files ...")
inp = Path(INPUT_DIR)
if not inp.exists():
    raise SystemExit(f"❌ INPUT_DIR not found: {inp}\n   Upload the dataset to Drive and fix INPUT_DIR at the top of the cell.")
out_root = Path(OUTPUT_DIR)
if out_root.resolve() == inp.resolve():
    raise SystemExit("❌ OUTPUT_DIR must be different from INPUT_DIR.")
files = sorted(p for p in inp.rglob("*")
               if p.is_file() and p.suffix.lower() in AUDIO_EXT and out_root not in p.parents)
if not files:
    raise SystemExit(f"❌ No audio files found in {inp} (looked for {sorted(AUDIO_EXT)})")
print(f"✅ STEP 3 COMPLETED: {len(files)} audio file(s) found")

# ---------- helpers ----------
def load_audio(path):
    try:
        x, sr = sf.read(str(path), dtype="float32", always_2d=True)
        x = x.mean(axis=1)
    except Exception:
        x, sr = librosa.load(str(path), sr=None, mono=True)   # mp3 / m4a via ffmpeg
    if sr != SR:
        x = librosa.resample(x, orig_sr=sr, target_sr=SR)
    return x.astype(np.float32)

def energy_db(x, frame=400, hop=160):
    n = max(1, 1 + (len(x) - frame) // hop)
    idx = np.minimum(np.arange(frame)[None, :] + hop * np.arange(n)[:, None], len(x) - 1)
    return 10 * np.log10(np.mean(x[idx] ** 2, axis=1) + 1e-10)

def find_cuts(x, target_s, min_s):
    """Cut near every target_s seconds, at a pause (quietest spot within +-30%)."""
    db, hop, total = energy_db(x), 160, len(x)
    target, min_len = int(target_s * SR), int(min_s * SR)
    window = int(0.3 * target)
    cuts, start = [0], 0
    while total - start > target + min_len:
        lo = start + target - window
        hi = min(start + target + window, total - min_len)
        f_lo, f_hi = lo // hop, max(lo // hop + 1, hi // hop)
        seg = db[f_lo:f_hi]
        if len(seg) > 20:
            seg = np.convolve(seg, np.ones(20) / 20, mode="same")
        quiet = np.where(seg <= seg.min() + 3.0)[0]
        want = (start + target) // hop - f_lo
        cut = (f_lo + int(quiet[np.argmin(np.abs(quiet - want))])) * hop
        cuts.append(cut); start = cut
    cuts.append(total)
    return cuts

def mostly_silent(clip):
    return np.mean(energy_db(clip) > -45) < 0.2

def save(path, x):
    peak = float(np.max(np.abs(x))) if len(x) else 0.0
    if peak > 1.0:
        x = x / peak
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), x, SR, subtype="PCM_16")

def chunk(x, seconds, out_dir, stem, tag):
    kept = []
    cuts = find_cuts(x, seconds, MIN_CLIP_SECONDS)
    for i in range(len(cuts) - 1):
        c = x[cuts[i]:cuts[i + 1]]
        if len(c) < MIN_CLIP_SECONDS * SR or mostly_silent(c):
            continue
        p = out_dir / f"{stem}_{tag}_{len(kept):03d}.wav"
        save(p, c); kept.append((p, c))
    return kept

# STEP 4: chunk / convert
print("▶ STEP 4/5: chunking ...")
long_clips, n_out, failed = [], 0, []
for k, f in enumerate(files, 1):
    rel = f.relative_to(inp)
    try:
        x = load_audio(f)
    except Exception as e:
        failed.append(f"{rel}: {e}"); print(f"  ⚠️ skipped {rel}: {e}"); continue
    dur = len(x) / SR
    if dur > LONG_FILE_SECONDS:
        d = out_root / "long_recordings" / rel.parent / f.stem
        c30 = chunk(x, CLIP_SECONDS_LONG, d / f"{CLIP_SECONDS_LONG}s", f.stem, f"{CLIP_SECONDS_LONG}s")
        c10 = chunk(x, CLIP_SECONDS_SHORT, d / f"{CLIP_SECONDS_SHORT}s", f.stem, f"{CLIP_SECONDS_SHORT}s")
        long_clips += c10; n_out += len(c30) + len(c10)
        print(f"  [{k}/{len(files)}] {rel}: {dur/60:.1f} min -> {len(c30)} x ~{CLIP_SECONDS_LONG}s + {len(c10)} x ~{CLIP_SECONDS_SHORT}s")
    else:
        if dur < 0.5:
            failed.append(f"{rel}: too short ({dur:.2f} s)"); continue
        save(out_root / "short_clips" / rel.parent / f"{f.stem}.wav", x); n_out += 1
        if k % 50 == 0 or k == len(files):
            print(f"  [{k}/{len(files)}] converted short clips ...")

if SPEAKER_NAME and long_clips:
    good = sorted(long_clips, key=lambda pc: -np.mean(energy_db(pc[1]) > -40))
    # spread picks over the recording: take every other of the best clips
    picks = [pc for pc in good if len(pc[1]) >= 5 * SR][:10][::2] or good[:5]
    for i, (_, c) in enumerate(picks[:5], 1):
        save(out_root / "speakers" / SPEAKER_NAME / f"{SPEAKER_NAME}_{i}.wav", c[: 10 * SR])
    print(f"  Registered-speaker clips: {min(5, len(picks))} -> speakers/{SPEAKER_NAME}/")
print(f"✅ STEP 4 COMPLETED: {n_out} clip(s) written to {out_root}" + (f", {len(failed)} file(s) failed" if failed else ""))

# STEP 5: zip for one download
print("▶ STEP 5/5: zipping ...")
if MAKE_ZIP:
    zpath = out_root.parent / f"{out_root.name}.zip"
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(out_root.rglob("*.wav")):
            z.write(p, p.relative_to(out_root.parent))
    print(f"✅ STEP 5 COMPLETED: {zpath} ({zpath.stat().st_size/1e6:.1f} MB)")
else:
    print("✅ STEP 5 COMPLETED: zip skipped")

print(f"\n🎉 DONE in {(time.time()-t0)/60:.1f} min.")
print("Folders in Drive:")
print(f"  {out_root}/long_recordings/<name>/30s/  -> ASR tests")
print(f"  {out_root}/long_recordings/<name>/10s/  -> AT / SV tests")
print(f"  {out_root}/short_clips/                 -> converted short AT files")
if SPEAKER_NAME:
    print(f"  {out_root}/speakers/{SPEAKER_NAME}/     -> copy into the web app's speakers/ folder")
if failed:
    print("Failed files:"); [print("  ", s) for s in failed[:20]]
